# Nghien Cuu He Thong Truy Xuat Tri Thuc (RAG) Cho Ung Dung Dat Mon Nha Hang

---

| | |
|---|---|
| **Tac gia** | [Ten sinh vien] |
| **Mon hoc** | Hoc may va Khai pha du lieu |
| **Giao vien** | [Ten giao vien] |
| **Ngay** | 2026-07-14 |

## Tom tat

Nghien cuu nay danh gia hieu qua cua **ba phuong phap truy xuat tri thuc** (Information Retrieval) trong he thong **RAG (Retrieval-Augmented Generation)** phuc vu chatbot dat mon nha hang:

1. **BM25** - Lexical retrieval (Okapi BM25 + Title/Tag Boosting)
2. **Dense Retrieval** - Semantic retrieval (`intfloat/multilingual-e5-small`, 384 dims)
3. **Hybrid RRF** - Reciprocal Rank Fusion ket hop BM25 + Dense

Thi nghiem tren **57 golden questions**, **147 knowledge chunks** tu 11 tai lieu.

**Metrics**: Hit@K, MRR@K, nDCG@K, Precision@K, Recall@K

**Kiem dinh thong ke**: Wilcoxon signed-rank, McNemar exact, Paired Bootstrap (10,000 iters), Holm-Bonferroni.


## 1. Dinh Nghia Bai Toan

### 1.1 Boi canh

He thong **CMC Restaurant AI Ordering** la chatbot ho tro khach hang dat mon tai nha hang thong qua giao dien QR code. Chatbot su dung kien truc **RAG (Retrieval-Augmented Generation)** de tra loi cau hoi dua tren kho tri thuc rieng cua nha hang.

### 1.2 Bai toan truy xuat (Information Retrieval)

**Cho:**
- Kho tri thuc $K = \{d_1, d_2, ..., d_N\}$ gom $N$ knowledge chunks
- Cau hoi $q$ tu khach hang
- Tap tai lieu lien quan $R(q) \subseteq K$ (ground truth)

**Muc tieu:**
- Tim ham ranking $f(q, K) \rightarrow [d_{\pi(1)}, d_{\pi(2)}, ..., d_{\pi(k)}]$
- **Toi uu**: tai lieu trong $R(q)$ xuat hien cang cao trong ranking cang tot

### 1.3 Tai sao can RAG?

| Van de | Giai phap RAG |
|---|---|
| LLM khong biet menu/gia/chinh sach rieng | RAG bo sung context tu kho tri thuc |
| LLM co the "bia" thong tin (hallucination) | RAG grounding gioi han LLM trong du lieu that |
| Menu thay doi theo mua/ngay | Cap nhat knowledge base, khong can retrain LLM |

### 1.4 Phuong phap so sanh

| Method | Loai | Mo ta |
|---|---|---|
| **BM25** | Lexical | Okapi BM25 voi title/tag boosting |
| **Dense** | Semantic | `intfloat/multilingual-e5-small` (384 dims, cosine similarity) |
| **Hybrid RRF** | Ket hop | Reciprocal Rank Fusion: BM25 + Dense |


In [1]:
# 0. IMPORTS VA CAU HINH
from __future__ import annotations
import csv, json, math, os, random, statistics, sys, time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Sequence
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.retrieval_metrics import (
    QueryMetrics, RetrievalSummary, evaluate_rankings, score_query, summarize_query_metrics,
)
from evaluation.statistical_tests import (
    holm_bonferroni, mcnemar_exact, paired_bootstrap, wilcoxon_signed_rank,
)
from app.rag.knowledge_base import KnowledgeChunk, load_markdown_knowledge_base
from app.rag.retriever import BM25Retriever, BM25_K1, BM25_B, TITLE_BOOST, TAG_BOOST

try:
    from app.rag.embedding_retriever import DenseRetriever, SentenceTransformerE5Encoder
    from app.rag.hybrid_retriever import HybridRrfRetriever
    HAS_DENSE = True
    print("sentence-transformers available")
except ImportError:
    HAS_DENSE = False
    print("sentence-transformers not installed. pip install sentence-transformers")

GOLDEN_QUESTIONS_PATH = PROJECT_ROOT / "evaluation" / "golden_questions.csv"
KNOWLEDGE_BASE_PATH = PROJECT_ROOT / "knowledge-base"
K_VALUES = (1, 3, 5, 10)
SEED = 20260714
BOOTSTRAP_ITERATIONS = 10_000
print(f"K={K_VALUES}, Bootstrap={BOOTSTRAP_ITERATIONS:,}, Seed={SEED}")


sentence-transformers available
K=(1, 3, 5, 10), Bootstrap=10,000, Seed=20260714


## 2. Mo Ta Du Lieu

### 2.1 Golden Questions (Test Set)

Bo test gom **60 cau hoi** duoc tao thu cong, moi cau co:
- `case_id`: ma dinh danh
- `user_question`: cau hoi tieng Viet mo phong khach hang thuc te
- `expected_sources`: danh sach tai lieu ground truth
- `expected_guardrail_flags`: co an toan (OUT_OF_SCOPE, PROFANITY, v.v.)

Sau khi loai 3 cau OUT_OF_SCOPE, con **57 cau** dung cho thi nghiem.

### 2.2 Knowledge Base (Corpus)

11 tai lieu Markdown, moi file duoc chia thanh chunks theo heading `#`.


In [2]:
# 2. LOAD VA PHAN TICH DU LIEU
def load_golden_questions(path):
    questions = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            sources = [s.strip() for s in row["expected_sources"].split(";") if s.strip()]
            if not sources:
                continue
            questions.append({
                "case_id": row["case_id"].strip(),
                "question": row["user_question"].strip(),
                "expected_sources": sources,
            })
    return questions

questions = load_golden_questions(GOLDEN_QUESTIONS_PATH)
chunks = load_markdown_knowledge_base(KNOWLEDGE_BASE_PATH)

print(f"Golden Questions: {len(questions)} (sau khi loai OUT_OF_SCOPE)")
print(f"Knowledge Chunks: {len(chunks)} tu {len(set(c.source for c in chunks))} files")

source_counter = Counter()
for q in questions:
    for s in q["expected_sources"]:
        source_counter[s] += 1

multi = sum(1 for q in questions if len(q["expected_sources"]) > 1)
print(f"\nSingle-source: {len(questions)-multi} ({(len(questions)-multi)/len(questions)*100:.0f}%)")
print(f"Multi-source:  {multi} ({multi/len(questions)*100:.0f}%)")
print("\nExpected sources distribution:")
for src, cnt in source_counter.most_common():
    bar = chr(9608) * int(cnt / len(questions) * 30)
    print(f"  {src:35s} {cnt:3d} ({cnt/len(questions)*100:5.1f}%) {bar}")


Golden Questions: 57 (sau khi loai OUT_OF_SCOPE)
Knowledge Chunks: 147 tu 11 files

Single-source: 23 (40%)
Multi-source:  34 (60%)

Expected sources distribution:
  menu.md                              25 (43.9%) #############
  faq.md                               16 (28.1%) ########
  restaurant-info.md                   10 (17.5%) #####
  combo-pairing.md                      9 (15.8%) ####
  ingredient-nutrition.md               7 (12.3%) ###
  allergy-dietary.md                    7 (12.3%) ###
  data-mining-insights.md               6 (10.5%) ###
  ordering-policy.md                    5 ( 8.8%) ##
  seasonal-promotion.md                 5 ( 8.8%) ##
  service-guide.md                      4 ( 7.0%) ##


In [3]:
# Thong ke Knowledge Base
chunks_per_source = Counter(c.source for c in chunks)
chunk_lengths = [len(c.content) for c in chunks]

print("Knowledge Base Statistics:")
print(f"  Total chunks:    {len(chunks)}")
print(f"  Avg length:      {statistics.mean(chunk_lengths):.0f} chars")
print(f"  Min length:      {min(chunk_lengths)} chars")
print(f"  Max length:      {max(chunk_lengths)} chars")
print(f"  Median:          {statistics.median(chunk_lengths):.0f} chars")
print("\nChunks per source:")
for src, cnt in sorted(chunks_per_source.items()):
    bar = chr(9608) * cnt
    print(f"  {src:35s} {cnt:3d} {bar}")


Knowledge Base Statistics:
  Total chunks:    147
  Avg length:      286 chars
  Min length:      56 chars
  Max length:      1151 chars
  Median:          228 chars

Chunks per source:
  allergy-dietary.md                   10 ##########
  brand-voice.md                       13 #############
  combo-pairing.md                     16 ################
  data-mining-insights.md              15 ###############
  faq.md                               35 ###################################
  ingredient-nutrition.md              11 ###########
  menu.md                              15 ###############
  ordering-policy.md                    4 ####
  restaurant-info.md                   10 ##########
  seasonal-promotion.md                 9 #########
  service-guide.md                      9 #########


## 3. Phuong Phap Nghien Cuu

### 3.1 BM25 (Best Matching 25) - Okapi BM25

**Cong thuc:**

$$\text{Score}(q, D) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{tf}(t,D) \cdot (k_1 + 1)}{\text{tf}(t,D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

| Tham so | Gia tri | Y nghia |
|---|---|---|
| $k_1$ | 1.5 | Saturation parameter |
| $b$ | 0.75 | Length normalization |
| Title boost | 1.5 | Bonus cho match trong tieu de |
| Tag boost | 1.0 | Bonus cho match trong tags |

$$\text{Final} = \text{BM25} + 1.5 \times |q \cap \text{title}| + 1.0 \times |q \cap \text{tags}|$$

| Uu diem | Nhuoc diem |
|---|---|
| Nhanh (~1ms/query) | Khong hieu synonyms |
| Khong can pretrained model | Khong hieu paraphrase |
| Hieu qua voi exact match | Phu thuoc tu vung chinh xac |

### 3.2 Dense Retrieval - multilingual-e5-small

| Tham so | Gia tri |
|---|---|
| Model | `intfloat/multilingual-e5-small` |
| Dimension | 384 |
| Query prefix | `"query: {q}"` |
| Doc prefix | `"passage: {title} {content}"` |

$$\text{Score}(q, D) = \cos(\text{Enc}_q(q),\ \text{Enc}_d(D)) = \frac{\text{Enc}_q(q) \cdot \text{Enc}_d(D)}{\|\text{Enc}_q(q)\| \times \|\text{Enc}_d(D)\|}$$

| Uu diem | Nhuoc diem |
|---|---|
| Hieu ngu nghia, synonyms | Cham (~472ms/query) |
| Cross-lingual | Can GPU/CPU tinh toan |
| Paraphrase matching | Phu thuoc pretrained model |

### 3.3 Hybrid RRF - Reciprocal Rank Fusion

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{w_r}{k + \text{rank}_r(d)}, \quad k = 60$$

**Pipeline**: BM25 ranking $R_1$ + Dense ranking $R_2$ -> RRF fusion -> final ranking

### 3.4 Metrics Danh Gia

| Metric | Cong thuc | Y nghia |
|---|---|---|
| **Hit@K** | $\mathbb{1}[\|R(q) \cap \text{top}_K\| > 0]$ | Co relevant doc trong top-K? |
| **MRR@K** | $\frac{1}{\|Q\|} \sum_q \frac{1}{\text{rank}_\text{first}}$ | Vi tri relevant doc dau tien |
| **nDCG@K** | $\frac{\text{DCG}@K}{\text{IDCG}@K}$ | Chat luong ranking |
| **Precision@K** | $\frac{\|R(q) \cap \text{top}_K\|}{K}$ | Ty le relevant trong top-K |
| **Recall@K** | $\frac{\|R(q) \cap \text{top}_K\|}{\|R(q)\|}$ | Coverage |

### 3.5 Kiem Dinh Thong Ke

| Test | Input | H0 |
|---|---|---|
| **Wilcoxon Signed-Rank** | Paired MRR | median(A - B) = 0 |
| **McNemar Exact** | Binary Hit | P(A hit, B miss) = P(A miss, B hit) |
| **Paired Bootstrap** | nDCG, 10K iters | Mean delta = 0 |
| **Holm-Bonferroni** | All p-values | FWER correction |


In [4]:
# 4. THIET LAP THI NGHIEM
def run_retriever(retriever, questions, top_k=10):
    rankings = {}
    for q in questions:
        results = retriever.search(q["question"], top_k)
        seen = set()
        sources = []
        for r in results:
            if r.chunk.source not in seen:
                seen.add(r.chunk.source)
                sources.append(r.chunk.source)
        rankings[q["case_id"]] = sources
    return rankings

expected_map = {q["case_id"]: q["expected_sources"] for q in questions}
summaries = {}
per_query_results = {}
print(f"Corpus:     {len(chunks)} chunks")
print(f"Queries:    {len(questions)} golden questions")
print(f"K values:   {K_VALUES}")


Corpus:     147 chunks
Queries:    57 golden questions
K values:   (1, 3, 5, 10)


In [5]:
# 4.1 BM25 Retriever
print("Running BM25...")
t0 = time.perf_counter()
bm25 = BM25Retriever(chunks)
bm25_rankings = run_retriever(bm25, questions, top_k=max(K_VALUES))
bm25_time = time.perf_counter() - t0

bm25_summary, bm25_pq = evaluate_rankings(bm25_rankings, expected_map, k_values=K_VALUES)
summaries["BM25"] = bm25_summary
per_query_results["BM25"] = list(bm25_pq)

print(f"BM25: {bm25_time:.3f}s ({bm25_time*1000/len(questions):.1f} ms/query)")
for k in K_VALUES:
    m = bm25_summary.by_k[k]
    print(f"  K={k:2d}: Hit={m.hit_rate:.4f}  MRR={m.mrr:.4f}  nDCG={m.ndcg:.4f}  Recall={m.recall:.4f}")


Running BM25...
BM25: 0.068s (1.2 ms/query)
  K= 1: Hit=0.6491  MRR=0.6491  nDCG=0.6491  Recall=0.3947
  K= 3: Hit=0.8421  MRR=0.7427  nDCG=0.6436  Recall=0.6784
  K= 5: Hit=0.8772  MRR=0.7506  nDCG=0.6900  Recall=0.7690
  K=10: Hit=0.9123  MRR=0.7560  nDCG=0.7098  Recall=0.8216


In [6]:
# 4.2 Dense Retriever (multilingual-e5-small)
if HAS_DENSE:
    print("Loading E5 encoder (intfloat/multilingual-e5-small)...")
    t0 = time.perf_counter()
    encoder = SentenceTransformerE5Encoder()
    print(f"Model loaded in {time.perf_counter()-t0:.1f}s (dim={encoder.dimension})")

    print("Running Dense retrieval...")
    t0 = time.perf_counter()
    dense = DenseRetriever(chunks, encoder)
    dense_rankings = run_retriever(dense, questions, top_k=max(K_VALUES))
    dense_time = time.perf_counter() - t0
    dense_summary, dense_pq = evaluate_rankings(dense_rankings, expected_map, k_values=K_VALUES)
    summaries["Dense"] = dense_summary
    per_query_results["Dense"] = list(dense_pq)
    print(f"Dense: {dense_time:.3f}s ({dense_time*1000/len(questions):.1f} ms/query)")
    for k in K_VALUES:
        m = dense_summary.by_k[k]
        print(f"  K={k:2d}: Hit={m.hit_rate:.4f}  MRR={m.mrr:.4f}  nDCG={m.ndcg:.4f}  Recall={m.recall:.4f}")
else:
    print("Dense not available. Install: pip install sentence-transformers")


Loading E5 encoder (intfloat/multilingual-e5-small)...
Model loaded in 52.2s (dim=384)
Running Dense retrieval...
Dense: 26.920s (472.3 ms/query)
  K= 1: Hit=0.5965  MRR=0.5965  nDCG=0.5965  Recall=0.3421
  K= 3: Hit=0.9123  MRR=0.7339  nDCG=0.7122  Recall=0.8129
  K= 5: Hit=0.9298  MRR=0.7383  nDCG=0.7466  Recall=0.8830
  K=10: Hit=0.9298  MRR=0.7383  nDCG=0.7534  Recall=0.8977


In [7]:
# 4.3 Hybrid RRF (BM25 + Dense)
if HAS_DENSE:
    print("Running Hybrid RRF (k=60)...")
    t0 = time.perf_counter()
    hybrid = HybridRrfRetriever([bm25, dense], rrf_k=60)
    hybrid_rankings = run_retriever(hybrid, questions, top_k=max(K_VALUES))
    hybrid_time = time.perf_counter() - t0
    hybrid_summary, hybrid_pq = evaluate_rankings(hybrid_rankings, expected_map, k_values=K_VALUES)
    summaries["Hybrid"] = hybrid_summary
    per_query_results["Hybrid"] = list(hybrid_pq)
    print(f"Hybrid: {hybrid_time:.3f}s ({hybrid_time*1000/len(questions):.1f} ms/query)")
    for k in K_VALUES:
        m = hybrid_summary.by_k[k]
        print(f"  K={k:2d}: Hit={m.hit_rate:.4f}  MRR={m.mrr:.4f}  nDCG={m.ndcg:.4f}  Recall={m.recall:.4f}")

    print(f"\nLatency Summary:")
    print(f"  BM25:   {bm25_time*1000/len(questions):8.1f} ms/query")
    print(f"  Dense:  {dense_time*1000/len(questions):8.1f} ms/query")
    print(f"  Hybrid: {hybrid_time*1000/len(questions):8.1f} ms/query")


Running Hybrid RRF (k=60)...
Hybrid: 1.126s (19.8 ms/query)
  K= 1: Hit=0.5789  MRR=0.5789  nDCG=0.5789  Recall=0.3421
  K= 3: Hit=0.8947  MRR=0.7251  nDCG=0.6993  Recall=0.7865
  K= 5: Hit=0.9123  MRR=0.7287  nDCG=0.7422  Recall=0.8743
  K=10: Hit=1.0000  MRR=0.7400  nDCG=0.7866  Recall=1.0000

Latency Summary:
  BM25:        1.2 ms/query
  Dense:     472.3 ms/query
  Hybrid:     19.8 ms/query


## 5. Ket Qua Thi Nghiem


In [8]:
# 5. BANG SO SANH TONG HOP
methods = list(summaries.keys())
for k in K_VALUES:
    print(f"\n{'='*75}")
    print(f"  K = {k}")
    print(f"{'='*75}")
    print(f"  {'Method':12s} | {'Hit@K':>8s} | {'MRR@K':>8s} | {'nDCG@K':>8s} | {'Prec@K':>8s} | {'Recall@K':>8s}")
    print(f"  {'-'*12} | {'-'*8} | {'-'*8} | {'-'*8} | {'-'*8} | {'-'*8}")
    for method in methods:
        m = summaries[method].by_k[k]
        print(f"  {method:12s} | {m.hit_rate:8.4f} | {m.mrr:8.4f} | {m.ndcg:8.4f} | {m.precision:8.4f} | {m.recall:8.4f}")
    bh = max(methods, key=lambda m: summaries[m].by_k[k].hit_rate)
    bm = max(methods, key=lambda m: summaries[m].by_k[k].mrr)
    bn = max(methods, key=lambda m: summaries[m].by_k[k].ndcg)
    print(f"  Best: Hit->{bh}, MRR->{bm}, nDCG->{bn}")


  K = 1
  Method       |    Hit@K |    MRR@K |   nDCG@K |   Prec@K | Recall@K
  ------------ | -------- | -------- | -------- | -------- | --------
  BM25         |   0.6491 |   0.6491 |   0.6491 |   0.6491 |   0.3947
  Dense        |   0.5965 |   0.5965 |   0.5965 |   0.5965 |   0.3421
  Hybrid       |   0.5789 |   0.5789 |   0.5789 |   0.5789 |   0.3421
  Best: Hit->BM25, MRR->BM25, nDCG->BM25

  K = 3
  Method       |    Hit@K |    MRR@K |   nDCG@K |   Prec@K | Recall@K
  ------------ | -------- | -------- | -------- | -------- | --------
  BM25         |   0.8421 |   0.7427 |   0.6436 |   0.3626 |   0.6784
  Dense        |   0.9123 |   0.7339 |   0.7122 |   0.4503 |   0.8129
  Hybrid       |   0.8947 |   0.7251 |   0.6993 |   0.4386 |   0.7865
  Best: Hit->Dense, MRR->BM25, nDCG->Dense

  K = 5
  Method       |    Hit@K |    MRR@K |   nDCG@K |   Prec@K | Recall@K
  ------------ | -------- | -------- | -------- | -------- | --------
  BM25         |   0.8772 |   0.7506 |   0.6900 | 

## 6. Phan Tich Thong Ke

Kiem dinh: **"Co su khac biet co y nghia thong ke giua cac methods khong?"**

- $\alpha = 0.05$, Holm-Bonferroni correction


In [9]:
# 6. KIEM DINH THONG KE
if len(summaries) >= 2:
    methods = list(per_query_results.keys())
    for test_k in [1, 3, 5]:
        print(f"\n{'='*75}")
        print(f"  STATISTICAL TESTS AT K = {test_k}")
        print(f"{'='*75}")
        p_values_raw = {}
        for i in range(len(methods)):
            for j in range(i+1, len(methods)):
                a, b = methods[i], methods[j]
                a_m, b_m = per_query_results[a], per_query_results[b]
                a_mrr = [x.by_k[test_k].reciprocal_rank for x in a_m]
                b_mrr = [x.by_k[test_k].reciprocal_rank for x in b_m]
                a_ndcg = [x.by_k[test_k].ndcg for x in a_m]
                b_ndcg = [x.by_k[test_k].ndcg for x in b_m]
                a_hit = [x.by_k[test_k].hit > 0 for x in a_m]
                b_hit = [x.by_k[test_k].hit > 0 for x in b_m]
                pair = f"{a} vs {b}"
                print(f"\n  --- {pair} ---")
                wsr = wilcoxon_signed_rank(a_mrr, b_mrr)
                print(f"  Wilcoxon (MRR):  p={wsr.p_value:.6f}, r_rb={wsr.rank_biserial:+.4f}")
                p_values_raw[f"Wilcoxon ({pair})"] = wsr.p_value
                bs = paired_bootstrap(a_ndcg, b_ndcg, iterations=BOOTSTRAP_ITERATIONS, seed=SEED)
                print(f"  Bootstrap (nDCG): p={bs.p_value:.6f}, delta={bs.mean_delta:+.6f}, CI=[{bs.ci_lower:+.6f}, {bs.ci_upper:+.6f}]")
                p_values_raw[f"Bootstrap ({pair})"] = bs.p_value
                mc = mcnemar_exact(a_hit, b_hit)
                print(f"  McNemar (Hit):   p={mc.p_value:.6f}")
                p_values_raw[f"McNemar ({pair})"] = mc.p_value
        adjusted = holm_bonferroni(p_values_raw)
        print(f"\n  Holm-Bonferroni ({len(adjusted)} tests):")
        for name, adj_p in sorted(adjusted.items(), key=lambda x: x[1]):
            v = "REJECT H0" if adj_p < 0.05 else "FAIL TO REJECT"
            print(f"    {name:40s} p_adj={adj_p:.6f} [{v}]")


  STATISTICAL TESTS AT K = 1

  --- BM25 vs Dense ---
  Wilcoxon (MRR):  p=0.581055, r_rb=+0.2308
  Bootstrap (nDCG): p=0.531747, delta=+0.052632, CI=[-0.063158, +0.175439]
  McNemar (Hit):   p=0.581055

  --- BM25 vs Hybrid ---
  Wilcoxon (MRR):  p=0.454834, r_rb=+0.3333
  Bootstrap (nDCG): p=0.384162, delta=+0.070175, CI=[-0.052632, +0.192982]
  McNemar (Hit):   p=0.454834

  --- Dense vs Hybrid ---
  Wilcoxon (MRR):  p=1.000000, r_rb=+0.0000
  Bootstrap (nDCG): p=0.872113, delta=+0.017544, CI=[-0.087719, +0.122807]
  McNemar (Hit):   p=1.000000

  Holm-Bonferroni (9 tests):
    Bootstrap (BM25 vs Hybrid)           p_adj=1.000000 [FAIL TO REJECT]
    Wilcoxon (BM25 vs Hybrid)            p_adj=1.000000 [FAIL TO REJECT]
    McNemar (BM25 vs Hybrid)             p_adj=1.000000 [FAIL TO REJECT]
    Bootstrap (BM25 vs Dense)            p_adj=1.000000 [FAIL TO REJECT]
    Wilcoxon (BM25 vs Dense)             p_adj=1.000000 [FAIL TO REJECT]
    McNemar (BM25 vs Dense)              p_adj=1.00

## 7. Phan Tich Loi (Error Analysis)


In [10]:
# 7. ERROR ANALYSIS tai K=3
ak = 3
methods = list(per_query_results.keys())
q_map = {q["case_id"]: q for q in questions}
all_fail, m_wins, m_fails = [], {m: [] for m in methods}, {m: [] for m in methods}
for idx, cid in enumerate(q_map):
    hits = {m: (idx < len(per_query_results[m]) and per_query_results[m][idx].by_k[ak].hit > 0) for m in methods}
    if not any(hits.values()):
        all_fail.append(cid)
    for m in methods:
        if hits[m] and not any(hits.get(o, False) for o in methods if o != m):
            m_wins[m].append(cid)
        if not hits[m] and all(hits.get(o, False) for o in methods if o != m):
            m_fails[m].append(cid)
print(f"Error Analysis at K={ak}")
print(f"\nAll methods failed: {len(all_fail)}")
for cid in all_fail:
    print(f'  x {cid}: "{q_map[cid]["question"]}"  Expected: {q_map[cid]["expected_sources"]}')
for m in methods:
    print(f"\n{m}: Unique wins={len(m_wins[m])}, Unique fails={len(m_fails[m])}")
    for cid in m_wins[m][:5]:
        print(f'  + {cid}: "{q_map[cid]["question"][:60]}"')
    for cid in m_fails[m][:5]:
        print(f'  - {cid}: "{q_map[cid]["question"][:60]}"')
print(f"\nDifficulty distribution (K={ak}):")
diff = Counter()
for idx in range(len(questions)):
    n = sum(1 for m in methods if idx < len(per_query_results[m]) and per_query_results[m][idx].by_k[ak].hit > 0)
    diff[n] += 1
for n in range(len(methods)+1):
    c = diff.get(n, 0)
    print(f"  {n}/{len(methods)} hit: {c:3d} ({c/len(questions)*100:5.1f}%)")


Error Analysis at K=3

All methods failed: 4
  x rag_002: "Ban dat luon com suon cho toi nhe"  Expected: ['ordering-policy.md']
  x rag_011: "Gui don cho toi luon di"  Expected: ['ordering-policy.md']
  x rag_027: "Cho xem do uong"  Expected: ['menu.md']
  x rag_029: "Them Pho bo vao gio hang cho toi"  Expected: ['ordering-policy.md']

BM25: Unique wins=1, Unique fails=4
  + rag_023: "Tinh tien giup toi"
  - rag_003: "Co mon nao thanh mat khong?"
  - rag_026: "Toi an keto, goi y mon gi?"
  - rag_042: "Pho bo bao nhieu tien?"
  - rag_047: "Co mon nao protein cao khong?"

Dense: Unique wins=1, Unique fails=0
  + rag_054: "Tom hum gia bao nhieu?"

Hybrid: Unique wins=0, Unique fails=0

Difficulty distribution (K=3):
  0/3 hit:   4 (  7.0%)
  1/3 hit:   2 (  3.5%)
  2/3 hit:   4 (  7.0%)
  3/3 hit:  47 ( 82.5%)


## 8. Ket Luan va Huong Phat Trien


In [11]:
# 8. KET LUAN
methods = list(summaries.keys())
k = 3
bh = max(methods, key=lambda m: summaries[m].by_k[k].hit_rate)
bm = max(methods, key=lambda m: summaries[m].by_k[k].mrr)
bn = max(methods, key=lambda m: summaries[m].by_k[k].ndcg)
print(f"KET LUAN (K={k})")
print(f"  Best Hit@{k}:  {bh:12s} = {summaries[bh].by_k[k].hit_rate:.4f}")
print(f"  Best MRR@{k}:  {bm:12s} = {summaries[bm].by_k[k].mrr:.4f}")
print(f"  Best nDCG@{k}: {bn:12s} = {summaries[bn].by_k[k].ndcg:.4f}")
if len(methods) >= 3:
    h10 = {m: summaries[m].by_k[10] for m in methods}
    print(f"\n  K=10: Hybrid Hit@10={h10['Hybrid'].hit_rate:.4f}, Recall@10={h10['Hybrid'].recall:.4f}")
print(f"\nNHAN XET:")
print("  BM25 dan dau MRR (exact keyword match manh)")
print("  Dense dan dau Hit@3, nDCG@3 (semantic understanding)")
print("  Hybrid dat Hit@10=1.00, Recall@10=1.00 (khong bo sot)")
print("  Statistical tests: p > 0.05 (not significant) do corpus nho")
print("  -> Hybrid duoc chon cho production (recall toi da)")
print(f"\nHAN CHE:")
print(f"  Corpus nho ({len(questions)} queries)")
print("  Golden questions tao thu cong")
print("  Chua danh gia end-to-end generation")
print(f"\nHUONG PHAT TRIEN:")
print("  Tang dataset len 200+ queries")
print("  Fine-tune E5 tren restaurant domain")
print("  Cross-encoder re-ranking")
print("  End-to-end evaluation")


KET LUAN (K=3)
  Best Hit@3:  Dense        = 0.9123
  Best MRR@3:  BM25         = 0.7427
  Best nDCG@3: Dense        = 0.7122

  K=10: Hybrid Hit@10=1.0000, Recall@10=1.0000

NHAN XET:
  BM25 dan dau MRR (exact keyword match manh)
  Dense dan dau Hit@3, nDCG@3 (semantic understanding)
  Hybrid dat Hit@10=1.00, Recall@10=1.00 (khong bo sot)
  Statistical tests: p > 0.05 (not significant) do corpus nho
  -> Hybrid duoc chon cho production (recall toi da)

HAN CHE:
  Corpus nho (57 queries)
  Golden questions tao thu cong
  Chua danh gia end-to-end generation

HUONG PHAT TRIEN:
  Tang dataset len 200+ queries
  Fine-tune E5 tren restaurant domain
  Cross-encoder re-ranking
  End-to-end evaluation
